In [3]:
import gymnasium as gym  
import numpy as np
import pandas as pd
import torch
import torch.nn as nn 
from torch.distributions.normal import Normal
import random 
from tqdm import tqdm 


In [36]:
class Reinforce_agent:
    def __init__(self):
        self.env = gym.make("InvertedPendulum-v4")
        self.n_states = self.env.observation_space.shape[0]
        self.n_actions = self.env.action_space.shape[0]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.gamma = 0.99
        self.lr = 0.0001
        self.eps = 1e-6

        self.probs = []
        self.rewards = []

    def create_policy_net(self):
        class Policy_Network(nn.Module):
            def __init__(self, n_actions, n_states):
                super(Policy_Network,self).__init__() 

                hidden_layer_1 = 16
                hidden_layer_2 = 32

                self.base_net= nn.Sequential(
                    nn.Linear(n_states,hidden_layer_1),
                    nn.Tanh(),
                    nn.Linear(hidden_layer_1,hidden_layer_2),
                    nn.Tanh(),
                )

                self.policy_mean_net = nn.Sequential(
                    nn.Linear(hidden_layer_2,n_actions)
                )
                self.policy_std_net = nn.Sequential(
                    nn.Linear(hidden_layer_2, n_actions)
                )
            def forward(self, states):
                base_nn_output = self.base_net(states.float())
                action_means = self.policy_mean_net(base_nn_output)
                action_stds = torch.log(1 + torch.exp(self.policy_std_net(base_nn_output)))
                return action_means,action_stds
            
        self.policy_net = Policy_Network(self.n_actions,self.n_states).to(self.device)
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(),lr=self.lr)

    def get_action(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(self.device)

        action_means, action_stds = self.policy_net(state)

        dist = Normal(action_means[0], action_stds[0])

        action = dist.sample()

        log_prob = dist.log_prob(action).sum()
        self.probs.append(log_prob)

        return action.detach().cpu().numpy()
    
    def update(self):
        running_g = 0   
        gs = []

        for R in self.rewards[::-1]:
            running_g = R+self.gamma *running_g
            gs.insert(0,running_g)
        deltas = torch.tensor(gs).to(self.device)

        log_probs = torch.stack(self.probs).squeeze()
        loss = -torch.sum(log_probs*deltas).to(self.device)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.probs = []
        self.rewards = [] 

    def train(self,n_episodes = 5000, ):
        for seed in [1,2,3,4,5]:
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)

        for i in tqdm(range(n_episodes)):

            state,info = self.env.reset(seed = seed)

            done = False
            while not done:
                action = self.get_action(state)
                state,reward,terminated, truncated,info = self.env.step(action)
                done = terminated or truncated
                self.rewards.append(reward)
            self.update()


    def save_model(self):
        torch.save(self.policy_net.state_dict(),"policy_agent")
    
    def load_model(self):
        self.policy_net.load_state_dict(torch.load("policy_agent"))
        self.policy_net.eval()


        


In [37]:
agent = Reinforce_agent()

c:\Anaconda\envs\rl\lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment InvertedPendulum-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


In [38]:
agent.create_policy_net()

In [39]:
agent.train()

100%|██████████| 5000/5000 [38:39<00:00,  2.16it/s]


In [ ]:
env = gym.make("InvertedPendulum-v4")
for i in range(10):
    done = False
    state , info = env.reset()

    while not done:
        action = agent.get_action(state)
        state, reward, terminated , truncated , info = env.step(action)
        done = terminated or truncated
env.close()

    

